# ABC Sonification Demo (abc2midi)

Uses the native `abc2midi` tool for ABC→MIDI conversion, then delegates to `midi_sonify` for synthesis.

Mirrors `abc_sonify_demo.ipynb` but with correct tempo, fermata, and instrument handling.

Uses the hymn **"Look Down, O Lord, From Heaven Behold"** (4-voice SATB arrangement).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import abc2midi_sonify as abc

## 1. Load ABC and inspect metadata

In [ ]:
ABC_PATH = "../data/hymns/Look_Down_O_Lord_From_Heaven_Behold-Ach_Gott_vom_Himmel.abc"
score = abc.load_abc(ABC_PATH)

meta = abc.get_metadata(score)
for k, v in meta.items():
    print(f"{k:>20}: {v}")

## 2. List parts with voice classifications

In [ ]:
parts = abc.list_parts(score)

header = f"{'Idx':>3}  {'Name':<10} {'Voice':<10} {'Notes':>5}  {'Pitch':>9}  {'Mean':>5}"
print(header)
print("-" * len(header))
for p in parts:
    pmin, pmax = p["pitch_min"], p["pitch_max"]
    pitch_str = f"{pmin:>3}–{pmax:<3}" if pmin is not None else "   –   "
    print(
        f"{p['instrument_index']:>3}  {p['name']:<10} {p['voice']:<10} "
        f"{p['note_count']:>5}  {pitch_str}  {p['mean_pitch']:>5.1f}"
    )

## 3. Select specific voices

In [ ]:
# Select by voice label
soprano = abc.select_parts(score, "Soprano")
print("Soprano parts:")
for p in abc.list_parts(soprano):
    print(f"  [{p['instrument_index']}] {p['name']} — {p['note_count']} notes, voice={p['voice']}")

print()

bass = abc.select_parts(score, "Bass")
print("Bass parts:")
for p in abc.list_parts(bass):
    print(f"  [{p['instrument_index']}] {p['name']} — {p['note_count']} notes, voice={p['voice']}")

print()

# Select by voice ID
s1v1 = abc.select_parts(score, "S1V1")
print("S1V1:")
for p in abc.list_parts(s1v1):
    print(f"  [{p['instrument_index']}] {p['name']} — {p['note_count']} notes, voice={p['voice']}")

print()

# Select by index
first_two = abc.select_parts(score, [0, 1])
print("Parts [0, 1]:")
for p in abc.list_parts(first_two):
    print(f"  [{p['instrument_index']}] {p['name']} — {p['note_count']} notes, voice={p['voice']}")

## 4. Convert to PrettyMIDI and show instrument mapping

In [ ]:
pm = abc.abc_to_midi(score)
print(f"Instruments: {len(pm.instruments)}")
print(f"Duration:    {pm.get_end_time():.1f} s")
print(f"Tempo:       {pm.estimate_tempo():.0f} BPM")
print()

instruments = abc.list_parts(score)
header = f"{'Idx':>3}  {'Name':<16} {'Prog':>4}  {'Notes':>5}  {'Pitch':>9}  {'Time':>13}  {'Poly':>5}"
print(header)
print("-" * len(header))
for info in instruments:
    pmin = info["pitch_min"]
    pmax = info["pitch_max"]
    pitch_str = f"{pmin:>3}–{pmax:<3}" if pmin is not None else "   –   "
    tstart = info["start_time"]
    tend = info["end_time"]
    time_str = f"{tstart:5.1f}–{tend:5.1f}" if tstart is not None else "     –     "
    print(
        f"{info['instrument_index']:>3}  {info['name']:<16} {info['program']:>4}  "
        f"{info['note_count']:>5}  {pitch_str}  {time_str}  "
        f"{info['approximate_polyphony']:>5.1f}"
    )

## 5. Synthesize full piece — sine-wave playback

In [ ]:
audio_full = abc.synthesize(score)
print(f"Audio shape: {audio_full.shape}, duration: {len(audio_full)/44100:.1f} s")
abc.play_audio(audio_full)

## 6. Trim by time and by measures

In [ ]:
pm_10s = abc.trim_time(score, 0.0, 10.0)
audio_10s = abc.synthesize(pm_10s)
print(f"Trimmed to 10 s — audio length: {len(audio_10s)/44100:.1f} s")
abc.play_audio(audio_10s)

In [ ]:
mtimes = abc.estimate_measure_times(score)
print(f"Total measures: {len(mtimes)}")
print("First 8 measure start times (s):")
for i, t in enumerate(mtimes[:8], 1):
    print(f"  Measure {i}: {t:.2f} s")

pm_m1_4 = abc.trim_measures(score, 1, 4)
audio_m1_4 = abc.synthesize(pm_m1_4)
print(f"\nMeasures 1–4 — audio length: {len(audio_m1_4)/44100:.1f} s")
abc.play_audio(audio_m1_4)

## 7. Chain: select voice + trim + synthesize

In [ ]:
soprano_score = abc.select_parts(score, "Soprano")
soprano_15s = abc.trim_time(soprano_score, 0.0, 15.0)
audio_soprano = abc.synthesize(soprano_15s)
print(f"Soprano first 15 s — audio length: {len(audio_soprano)/44100:.1f} s")
abc.play_audio(audio_soprano)

In [ ]:
bass_score = abc.select_parts(score, "Bass")
bass_m1_8 = abc.trim_measures(bass_score, 1, 8)
audio_bass = abc.synthesize(bass_m1_8)
print(f"Bass measures 1–8 — audio length: {len(audio_bass)/44100:.1f} s")
abc.play_audio(audio_bass)

## 8. FluidSynth comparison — sine vs SoundFont

In [ ]:
from IPython.display import display, HTML

SF2_PATH = "../data/soundfonts/GeneralUser_GS.sf2"
score_10s = abc.trim_time(score, 0.0, 10.0)

display(HTML("<b>Sine-wave synthesis:</b>"))
audio_sine = abc.synthesize(score_10s)
display(abc.play_audio(audio_sine))

display(HTML("<b>FluidSynth + SoundFont:</b>"))
audio_sf = abc.synthesize(score_10s, sf2_path=SF2_PATH)
display(abc.play_audio(audio_sf))

In [ ]:
audio_fluid_full = abc.synthesize(score, sf2_path=SF2_PATH)
print(f"FluidSynth full piece: {len(audio_fluid_full)/44100:.1f} s")
abc.play_audio(audio_fluid_full)

## 9. Extract and display lyrics

In [ ]:
# get_lyrics accepts an ABCScore or a path
lyrics = abc.get_lyrics(score)

if lyrics:
    current_verse = None
    for entry in lyrics:
        if entry["verse"] != current_verse:
            current_verse = entry["verse"]
            print(f"--- Verse {current_verse} ---")
        print(f"  [{entry['voice']}] {entry['text']}")
    print()
else:
    print("No lyrics found in the score.")

## 10. Sonify individual and combined parts

`sonify_part` — sonify a single voice by name, label, or index.
`sonify_parts` — pick any combination of voices and mix them together.

In [ ]:
from IPython.display import display, HTML

for label in ["Soprano", "Alto", "Tenor", "Bass"]:
    audio, sr = abc.sonify_part(score, label)
    display(HTML(f"<b>{label}:</b>"))
    display(abc.play_audio(audio, sr))

In [ ]:
display(HTML("<b>Soprano + Bass (outer voices):</b>"))
audio, sr = abc.sonify_parts(score, ["Soprano", "Bass"])
display(abc.play_audio(audio, sr))

display(HTML("<b>Soprano + Alto (upper voices):</b>"))
audio, sr = abc.sonify_parts(score, ["Soprano", "Alto"])
display(abc.play_audio(audio, sr))

display(HTML("<b>All four voices:</b>"))
audio, sr = abc.sonify_parts(score, ["Soprano", "Alto", "Tenor", "Bass"])
display(abc.play_audio(audio, sr))

In [ ]:
display(HTML("<b>S1V1 + S2V2 (FluidSynth):</b>"))
audio, sr = abc.sonify_parts(score, ["S1V1", "S2V2"], sf2_path=SF2_PATH)
display(abc.play_audio(audio, sr))